# Stage 2 — Pilot project + analyze (Qwen3-32B)

**Purpose:** sanity / preliminary transfer figures on a **subset** of normalized trajectories before the full program-A100 paper run.

**Runtime:** A100 80GB (bf16 32B).

**You will get (draft quality only):**
- `projections.parquet` (all layers)
- final-step separation + position SNR plots
- optional `agentic_meanpool_32b.npz` for Stage-5 probe fitting

**Not paper numbers** — replace with the full ~200-task run later.

**Pilot sample (default `TARGET_TOTAL=50`):**
- keep **all** successes (`outcome=1`)
- subsample failures (`outcome=0`) so total ≈ 50  
  (e.g. 30 succ + 170 fail → 30 succ + 20 fail)

If successes alone are already ≥ 50, still keep all of them and add at least one failure (so both classes exist).

**Requirements**
1. `value_axis_32b.npy` (and ideally `axis_manifest_32b.json`)
2. Zip of **normalized** traj JSONs with **both** outcomes (resolved + unresolved)
3. Mount Drive for outputs / resume-friendly artifacts

ETA: often **several hours** for ~50 trajs (one forward per step; scales with total steps).

In [ ]:
import torch
assert torch.cuda.is_available(), 'Need GPU (A100 80GB for Qwen3-32B bf16)'
print(torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import os
REPO = '/content/failure_prediction_research'
if not os.path.isdir(REPO):
    # Prefer your fork URL if different:
    !git clone https://github.com/abdelmagid07/failure_prediction_research.git {REPO}
else:
    %cd {REPO}
    !git pull
%cd {REPO}
!pip install -q -e stage1 -e stage2
!pip install -q pyarrow pandas scikit-learn matplotlib

In [ ]:
# Drive for outputs (survives disconnects). Projection itself is long — keep artifacts here.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/failure_prediction_research/stage2_pilot_32b')
OUT_DIR = DRIVE_ROOT / 'outputs'
ACT_DIR = DRIVE_ROOT / 'activations'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ACT_DIR.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('OUT_DIR:', OUT_DIR)

In [ ]:
# Pilot knobs
PRIMARY_LAYER = 49          # from Stage-1 32B axis (thinking ON)
MODEL = 'Qwen/Qwen3-32B'
N_LAYERS = 64
TARGET_TOTAL = 50           # keep ALL successes; fill remaining slots with failures
SEED = 0
SAVE_ACTIVATIONS = True     # REMINDER: needed for Stage-5 probes without a second GPU pass

print('PRIMARY_LAYER', PRIMARY_LAYER)
print('TARGET_TOTAL', TARGET_TOTAL)
print('SAVE_ACTIVATIONS', SAVE_ACTIVATIONS)

## Upload axis + normalized traj zip

Upload:
- `value_axis_32b.npy` (required)
- `axis_manifest_32b.json` (optional)
- a zip of **normalized** trajectory JSON files (from ingest)

The zip can contain nested folders; JSONs will be collected.

In [ ]:
import json, shutil, zipfile
from pathlib import Path
from google.colab import files

REPO = Path('/content/failure_prediction_research')
AXIS_DIR = REPO / 'stage1' / 'data'
NORM_DIR = REPO / 'stage2' / 'data' / 'normalized_pilot'
AXIS_DIR.mkdir(parents=True, exist_ok=True)
NORM_DIR.mkdir(parents=True, exist_ok=True)

print('Upload value_axis_32b.npy (+ optional axis_manifest_32b.json)...')
up = files.upload()
for name in up:
    dest = AXIS_DIR / Path(name).name
    shutil.move(name, dest)
    print('axis artifact ->', dest)

AXIS = AXIS_DIR / 'value_axis_32b.npy'
assert AXIS.exists(), f'Missing {AXIS}'

import numpy as np
axis = np.load(AXIS)
print('axis shape', axis.shape, '(expect 64 x 5120)')
assert axis.shape == (64, 5120), axis.shape

print('Upload normalized trajectories zip...')
zname = next(iter(files.upload()))
extract = REPO / 'stage2' / 'data' / '_pilot_zip_extract'
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir(parents=True)
with zipfile.ZipFile(zname) as z:
    z.extractall(extract)

# Flatten JSONs into NORM_DIR
for p in NORM_DIR.glob('*.json'):
    p.unlink()
n = 0
for p in extract.rglob('*.json'):
    shutil.copy2(p, NORM_DIR / p.name)
    n += 1
print(f'normalized JSONs available: {n} in {NORM_DIR}')

In [ ]:
# Keep ALL successes; subsample failures up to TARGET_TOTAL
import json, random, shutil
from pathlib import Path
from collections import Counter

REPO = Path('/content/failure_prediction_research')
NORM_DIR = REPO / 'stage2' / 'data' / 'normalized_pilot'
PILOT_DIR = REPO / 'stage2' / 'data' / 'normalized_pilot_selected'
if PILOT_DIR.exists():
    shutil.rmtree(PILOT_DIR)
PILOT_DIR.mkdir(parents=True)

files_json = sorted(NORM_DIR.glob('*.json'))
records = []
for p in files_json:
    d = json.loads(p.read_text())
    outcome = d.get('outcome')
    records.append((p, int(outcome), d.get('task_id'), d.get('n_steps') or len(d.get('steps', []))))

succ = [r for r in records if r[1] == 1]
fail = [r for r in records if r[1] == 0]
print('uploaded outcomes:', Counter(o for _, o, _, _ in records))
print(f'  successes={len(succ)}  failures={len(fail)}')
assert succ, 'Need at least one resolved (outcome=1)'
assert fail, 'Need at least one unresolved (outcome=0)'

rng = random.Random(SEED)
rng.shuffle(fail)

# Fill remaining slots with failures. If successes already >= TARGET_TOTAL,
# still keep ALL successes and take at least 1 failure (both classes).
n_fail_target = max(TARGET_TOTAL - len(succ), 1)
n_fail_target = min(n_fail_target, len(fail))
selected = list(succ) + fail[:n_fail_target]
rng.shuffle(selected)

for p, o, tid, ns in selected:
    shutil.copy2(p, PILOT_DIR / p.name)

print(f'pilot rule: keep all {len(succ)} successes + {n_fail_target}/{len(fail)} failures')
print(f'pilot trajs: {len(selected)} (TARGET_TOTAL={TARGET_TOTAL})')
print('pilot outcomes:', Counter(o for _, o, _, _ in selected))
print('mean n_steps:', round(sum(ns for *_, ns in selected) / max(len(selected), 1), 1))
print('PILOT_DIR:', PILOT_DIR)

## Project (GPU)

Projects **all layers** (forward already hooks all layers; restricting `--layers` barely helps).

**Reminder:** `SAVE_ACTIVATIONS=True` writes `--activations-npz` for Stage-5 probes without a second GPU pass.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
PILOT_DIR = REPO / 'stage2' / 'data' / 'normalized_pilot_selected'
AXIS = REPO / 'stage1' / 'data' / 'value_axis_32b.npy'
PROJ = OUT_DIR / 'projections_pilot.parquet'
ACT_NPZ = ACT_DIR / 'agentic_meanpool_32b_pilot.npz'

cmd = [
    sys.executable, '-u', '-m', 'stage2.extract.project_steps',
    '--traj-dir', str(PILOT_DIR),
    '--axis-path', str(AXIS),
    '--model', MODEL,
    '--n-layers', str(N_LAYERS),
    '--enable-thinking',
    '--output', str(PROJ),
]
if SAVE_ACTIVATIONS:
    cmd.extend(['--activations-npz', str(ACT_NPZ)])

print('CMD:', ' '.join(cmd), flush=True)
print('Quiet while loading Qwen3-32B is normal...', flush=True)

proc = subprocess.Popen(cmd, cwd=str(REPO / 'stage2'),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('project_steps exit:', rc, flush=True)
assert rc == 0 and PROJ.exists(), PROJ
print('projections ->', PROJ, 'size MB', round(PROJ.stat().st_size / 1e6, 2))
if SAVE_ACTIVATIONS:
    print('activations ->', ACT_NPZ, 'exists', ACT_NPZ.exists())

## Analyze (CPU-ish)

Headline plots use `PRIMARY_LAYER` (49).

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
PROJ = OUT_DIR / 'projections_pilot.parquet'
REPORT = OUT_DIR / 'analysis_report_pilot'
REPORT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', '-m', 'stage2.analyze.run_analyses',
    '--projections', str(PROJ),
    '--output-dir', str(REPORT),
    '--primary-layer', str(PRIMARY_LAYER),
]
print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, cwd=str(REPO / 'stage2'),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('run_analyses exit:', rc, flush=True)
assert rc == 0
print('report ->', REPORT)

## Optional: fit probes (Stage 5)

Only if `SAVE_ACTIVATIONS=True` and the npz exists. CPU-only; minutes–tens of minutes on a pilot.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
ACT_NPZ = ACT_DIR / 'agentic_meanpool_32b_pilot.npz'
PROBE_DIR = OUT_DIR / 'probe_report_pilot'

if not ACT_NPZ.exists():
    print('SKIP probes — missing', ACT_NPZ)
else:
    cmd = [
        sys.executable, '-u', '-m', 'stage2.probes.fit_probes',
        '--activations', str(ACT_NPZ),
        '--output-dir', str(PROBE_DIR),
    ]
    print('CMD:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=str(REPO / 'stage2'),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    print('fit_probes exit:', proc.wait(), flush=True)
    print('probe report ->', PROBE_DIR)

In [ ]:
# Preview headline metrics
import json
from pathlib import Path
from IPython.display import Image, display

REPORT = OUT_DIR / 'analysis_report_pilot'
rep = REPORT / 'analysis_report.json'
if rep.exists():
    print(json.dumps(json.loads(rep.read_text()), indent=2)[:3000])

for name in ['final_step_separation.png', 'noise_by_token_type.png']:
    p = REPORT / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
    else:
        print('missing', p)

In [ ]:
# Zip + download pilot artifacts
import zipfile
from pathlib import Path
from google.colab import files

zip_path = OUT_DIR / 'stage2_pilot_32b_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.json', '.png', '.csv', '.parquet'}:
            z.write(p, p.relative_to(OUT_DIR).as_posix())
    # activations can be large — include only if small enough / you want them
    act = ACT_DIR / 'agentic_meanpool_32b_pilot.npz'
    if act.exists() and act.stat().st_size < 1_500_000_000:
        z.write(act, f'activations/{act.name}')
    elif act.exists():
        print('activations npz too large for zip; left on Drive:', act)

print('zip ->', zip_path)
files.download(str(zip_path))